# Task 2: Model monitoring

In this notebook, you monitor and evaluate the data captured from the endpoint. You create a baseline with which you compare the real-time traffic. Once a baseline is ready, you set up a schedule to continuously evaluate and compare the data against the baseline.

## Task 2.1: Environment setup

In this task, you set up your environment.

In [1]:
#install-dependencies
%pip install "sagemaker>=2,<3"
%matplotlib inline
from datetime import datetime, timedelta
import json
import boto3
import time
import pandas as pd
import matplotlib.pyplot as plt
from sagemaker import get_execution_role, session
from sagemaker.s3 import S3Uploader
from sagemaker.image_uris import retrieve
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
from time import sleep
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

region = boto3.Session().region_name
role = get_execution_role()
sm_session = session.Session(boto3.Session())
sm = boto3.Session().client("sagemaker")
sm_runtime = boto3.Session().client("sagemaker-runtime")
cw = boto3.Session().client("cloudwatch")

#repo_path = './'
bucket = sm_session.default_bucket()
prefix = 'sagemaker/abalone'
data_capture_prefix = "{}/datacapture".format(prefix)
s3_capture_upload_path = "s3://{}/{}".format(bucket, data_capture_prefix)
capture_modes = [ "Input",  "Output" ]
code_prefix = "{}/code".format(prefix)
s3_code_preprocessor_uri = "s3://{}/{}/{}".format(bucket, code_prefix, "preprocessor.py")
s3_code_postprocessor_uri = "s3://{}/{}/{}".format(bucket, code_prefix, "postprocessor.py")
reports_prefix = "{}/reports".format(prefix)
s3_report_path = "s3://{}/{}".format(bucket, reports_prefix)

import os

# This looks for the 'models' folder in your current location or one level up/down
# to dynamically find the correct repo_path
current_dir = os.getcwd()

if not os.path.exists(os.path.join(current_dir, "models")):
    # Try to find the notebook's actual directory if the kernel is lost in root
    # In Jupyter, this usually resolves the folder containing the .ipynb
    repo_path = os.path.abspath('') 
    
    # If we are still in root, we force it to the notebook's path
    if repo_path == '/home/sagemaker-user':
        # This is a safe 'hack' to find the folder without hardcoding 'Lab5Repository'
        # It finds the first directory containing 'models'
        for root, dirs, files in os.walk('/home/sagemaker-user'):
            if 'models' in dirs and 'lab_7.ipynb' in files:
                repo_path = root
                break
else:
    repo_path = current_dir

print(f"Verified Repo Path: {repo_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.5 MB/s  0:00:00


  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [packaging]

  Attempting uninstall: attrs
    Found existing installation: attrs 26.1.0
    Uninstalling attrs-26.1.0:
      Successfully uninstalled attrs-26.1.0
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [packaging]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.11.0 requires sagemaker-core>=2.11.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytic

Note: you may need to restart the kernel to use updated packages.


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Verified Repo Path: /home/sagemaker-user/Lab7Repository/en_us


<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** You may ignore any warnings after running the above cell and thoroughout subsequent cells. 

## Task 2.2: Create a production endpoint with Data Capture enabled

To log the inputs to your endpoint and the inference outputs from your deployed model to Amazon S3, you can enable an Amazon SageMaker AI feature called Data Capture. Data Capture records information that can be used for training, debugging, and monitoring. SageMaker AI Model Monitor automatically parses this captured data and compares metrics from this data with a baseline that you create for the model.

In this task, you upload the pre-trained model to the S3 bucket, create an Sagemaker AI model object, configure a SageMaker AI real-time endpoint with Data Capture enabled, and create the real-time endpoint.

<i class="fas fa-sticky-note" style="color:#563377"></i> **Note:** Endpoint creation takes approximately 5 minutes to complete.

In [2]:
#create-production-endpoint
# Upload models
model_url = S3Uploader.upload(
    local_path=f"{repo_path}/models/model.tar.gz", desired_s3_uri=f"s3://{bucket}/{prefix}"
)

# Create the model definitions
model_name = f"abalone-A-{datetime.now():%Y-%m-%d-%H-%M-%S}"
image_uri = retrieve("xgboost", boto3.Session().region_name, "1.5-1")

# Create production model object
predictor=sm_session.create_model(
    name=model_name, role=role, container_defs={"Image": image_uri, "ModelDataUrl": model_url}
)

# Create the endpoint configurations
variant_name = 'AllTraffic'

endpoint_config_name = f'Abalone-Endpoint-1-{datetime.now():%Y-%m-%d-%H-%M-%S}'
endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName = endpoint_config_name,
    ProductionVariants=[
        {
            'ModelName':model_name,
            'InstanceType':'ml.m5.xlarge',
            'InitialInstanceCount':1,
            'VariantName':variant_name
        }
    ],
    
        DataCaptureConfig= {
        'EnableCapture': True, # Whether data should be captured or not.
        'InitialSamplingPercentage' : 100,
        'CaptureContentTypeHeader': {'CsvContentTypes': [ 'text/csv' ]},
        'DestinationS3Uri': s3_capture_upload_path,
        'CaptureOptions': [{"CaptureMode" : capture_mode} for capture_mode in capture_modes] # Example - Use list comprehension to capture both Input and Output
    }
)
print(f"Created the Production Model Endpoint Config: {endpoint_config_name}")
time.sleep(5)

# Create the endpoint with the production model
endpoint_name = f"Abalone-{datetime.now():%Y-%m-%d-%H-%M-%S}"
endpoint_response = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)

def wait_for_endpoint_creation_complete(endpoint):
    """Helper function to wait for the completion of creating an endpoint"""
    response = sm.describe_endpoint(EndpointName=endpoint_name)
    status = response.get("EndpointStatus")
    while status == "Creating":
        print("Waiting for Endpoint Creation")
        time.sleep(15)
        response = sm.describe_endpoint(EndpointName=endpoint_name)
        status = response.get("EndpointStatus")

    if status != "InService":
        print(f"Failed to create endpoint, response: {response}")
        failureReason = response.get("FailureReason", "")
        raise SystemExit(
            f"Failed to create endpoint {endpoint_response['EndpointArn']}, status: {status}, reason: {failureReason}"
        )
    print(f"Endpoint {endpoint_response['EndpointArn']} successfully created.")

wait_for_endpoint_creation_complete(endpoint=endpoint_response)

Created the Production Model Endpoint Config: Abalone-Endpoint-1-2026-06-12-10-28-33


Waiting for Endpoint Creation


Waiting for Endpoint Creation


Waiting for Endpoint Creation


Waiting for Endpoint Creation


...

Waiting for Endpoint Creation


Waiting for Endpoint Creation


Waiting for Endpoint Creation


Waiting for Endpoint Creation


Endpoint arn:aws:sagemaker:us-west-2:105806319240:endpoint/Abalone-2026-06-12-10-28-38 successfully created.


When the cell completes, an endpoint ARN is returned that looks like *arn:aws:sagemaker:us-west-2:012345678910:endpoint/abalone-2040-10-11-10-11-12*.

Your endpoint is currently configured with one variant, the production model. You can view the endpoint configuration using *describe_endpoint*.

In [3]:
#describe-the-endpoint
sm.describe_endpoint(EndpointName=endpoint_name)

{'EndpointName': 'Abalone-2026-06-12-10-28-38',
 'EndpointArn': 'arn:aws:sagemaker:us-west-2:105806319240:endpoint/Abalone-2026-06-12-10-28-38',
 'EndpointConfigName': 'Abalone-Endpoint-1-2026-06-12-10-28-33',
 'ProductionVariants': [{'VariantName': 'AllTraffic',
   'DeployedImages': [{'SpecifiedImage': '246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.5-1',
     'ResolvedImage': '246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost@sha256:cd128bd02824075bf2e02ee7923aaa8ab50f6c47a2d86d5747b78ca1c5199813',
     'ResolutionTime': datetime.datetime(2026, 6, 12, 10, 28, 40, 75000, tzinfo=tzlocal())}],
   'CurrentWeight': 1.0,
   'DesiredWeight': 1.0,
   'CurrentInstanceCount': 1,
   'DesiredInstanceCount': 1}],
 'DataCaptureConfig': {'EnableCapture': True,
  'CaptureStatus': 'Started',
  'CurrentSamplingPercentage': 100,
  'DestinationS3Uri': 's3://sagemaker-us-west-2-105806319240/sagemaker/abalone/datacapture'},
 'EndpointStatus': 'InService',
 'CreationTime': da

## Task 2.3: View the captured data

In this task, you invoke the endpoint created above using production data. Since you already enabled Data Capture on the endpoint, the request payload, response, and additional metadata is saved in the S3 location you specified earlier in the notebook. After invoking the endpoint, you examine the data captured in the S3 bucket.

First, use an initialized predictor configured with the endpoint name to invoke the endpoint. Invoking the endpoint with new records helps you confirm your Data Capture configuration is set up correctly. A predictor makes prediction requests to an Amazon SageMaker AI endpoint. Then, run inference by sending records to the endpoint.

In [4]:
#invoke-the-endpoint
predictor = Predictor(endpoint_name=endpoint_name,
                        serializer=CSVSerializer(),
                        deserializer=CSVDeserializer())

validate_dataset = "abalone_data_new_predictions.csv"

limit = 200  # Need at least 200 samples to compute standard deviations
i = 0
with open(f"{repo_path}/data/{validate_dataset}", "w") as validation_file:
    validation_file.write("prediction,label\n")  # CSV header
    with open(f"{repo_path}/data/abalone_data_new.csv", "r") as f:
        for row in f:
            (label, input_cols) = row.split(",", 1)
            prediction = predictor.predict(input_cols)[0][0]
            validation_file.write(f"{prediction},{label}\n")
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            sleep(0.5)

print("\nDone!")

.

.

.

.

.

...

.

.

.

.


Done!


Next, list the Data Capture files stored in Amazon S3.

In [5]:
#list-data-capture-files
s3_client = boto3.Session().client("s3")
current_endpoint_capture_prefix = "{}/{}".format(data_capture_prefix, endpoint_name)
result = s3_client.list_objects(Bucket=bucket, Prefix=current_endpoint_capture_prefix)
capture_files = [capture_file.get("Key") for capture_file in result.get("Contents")]
print("Found Capture Files:")
print("\n ".join(capture_files))

Found Capture Files:
sagemaker/abalone/datacapture/Abalone-2026-06-12-10-28-38/AllTraffic/2026/06/12/10/33-28-867-b7b86a39-1a5c-46b0-97dd-24f8da3dffc1.jsonl


Then, view the contents of a single Data Capture file. You should see all the data captured in a SageMaker AI specific JSON-line formatted file. Take a moment to review the first few lines in the captured file.

In [6]:
#view-captured-file-lines
def get_obj_body(obj_key):
    return s3_client.get_object(Bucket=bucket, Key=obj_key).get("Body").read().decode("utf-8")


capture_file = get_obj_body(capture_files[-1])
print(capture_file[:2000])

{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"0.8421737350064652,0.8609915417351095,0.6888110025132029,0.7977896490322643,0.8196374991984011,0.8545097875688132,0.4648058399193789,0.0,0.0,1.0\n","encoding":"CSV"},"endpointOutput":{"observedContentType":"text/csv; charset=utf-8","mode":"OUTPUT","data":"10.524738311767578\n","encoding":"CSV"}},"eventMetadata":{"eventId":"1d6d8121-88a9-4056-813d-394c930ae538","inferenceTime":"2026-06-12T10:33:28Z"},"eventVersion":"0"}
{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"1.403862929794872,1.523946529448224,1.6451766919530222,2.55695787544154,2.0651650715096967,2.736337747238168,2.1377931785031565,0.0,0.0,1.0\n","encoding":"CSV"},"endpointOutput":{"observedContentType":"text/csv; charset=utf-8","mode":"OUTPUT","data":"11.895867347717285\n","encoding":"CSV"}},"eventMetadata":{"eventId":"42b83fec-8032-48ec-b909-9e837af754ad","inferenceTime":"2026-06-12T10:33:29Z"},"

Finally, view the contents of one captured endpoint input and output record.

In [7]:
#print-json-file
print(json.dumps(json.loads(capture_file.split("\n")[0]), indent=2))

{
  "captureData": {
    "endpointInput": {
      "observedContentType": "text/csv",
      "mode": "INPUT",
      "data": "0.8421737350064652,0.8609915417351095,0.6888110025132029,0.7977896490322643,0.8196374991984011,0.8545097875688132,0.4648058399193789,0.0,0.0,1.0\n",
      "encoding": "CSV"
    },
    "endpointOutput": {
      "observedContentType": "text/csv; charset=utf-8",
      "mode": "OUTPUT",
      "data": "10.524738311767578\n",
      "encoding": "CSV"
    }
  },
  "eventMetadata": {
    "eventId": "1d6d8121-88a9-4056-813d-394c930ae538",
    "inferenceTime": "2026-06-12T10:33:28Z"
  },
  "eventVersion": "0"
}


Enabling Data Capture on your endpoints gives you more flexibility when you want to save information for training, debugging, and monitoring. Since SageMaker AI Model Monitor parses this captured data automatically, using Data Capture helps you compare new records to baseline data. 

You have not configured a baseline yet. In the next task, you use SageMaker AI Model Monitor to generate baseline statistics and constraints. 

## Task 2.4: Generate baseline statistics and constraints

In this task, you create a baseline. Baseline statistics and constraints serve as a standard for detecting data drift and other data quality issues. 

The test dataset from training the model is often a good baseline dataset. The test dataset schema and the inference dataset schema should exactly match, including the number and order of the features. From the test dataset, you can ask SageMaker AI to suggest a set of baseline constraints and generate descriptive statistics to explore the data.

First, configure the baseline prefixes and variables.

In [8]:
#configure-baseline-variables
baseline_prefix = prefix + "/baselining"
baseline_data_prefix = baseline_prefix + "/data"
baseline_results_prefix = baseline_prefix + "/results"

baseline_data_uri = "s3://{}/{}".format(bucket, baseline_data_prefix)
baseline_results_uri = "s3://{}/{}".format(bucket, baseline_results_prefix)
print("Baseline data uri: {}".format(baseline_data_uri))
print("Baseline results uri: {}".format(baseline_results_uri))

Baseline data uri: s3://sagemaker-us-west-2-105806319240/sagemaker/abalone/baselining/data
Baseline results uri: s3://sagemaker-us-west-2-105806319240/sagemaker/abalone/baselining/results


Next, start a job to suggest a baseline and constraints. *DefaultModelMonitor.suggest_baseline()* starts a **ProcessingJob** using a SageMaker AI provided Model Monitor container to generate the baseline and constraints.

<i class="fas fa-sticky-note" style="color:#563377"></i> **Note:** A baseline job takes approximately 10 minutes to complete.

<i class="fas fa-info-circle" style="color:#008296"></i> **Learn more:** Refer to [Create a Baseline](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-create-baseline.html) for more information about creating baseline calculations of statistics and constraints.

In [9]:
#create-baselining-job
my_default_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.t3.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

my_default_monitor.suggest_baseline(
    baseline_dataset=f"{repo_path}/data/abalone_data_new_withheader.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    wait=True,
)

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-12-10-35-42-126


.

.

.

.

...

2026-06-12 10:39:53,046 INFO yarn.Client: Application report for application_1781260764017_0001 (state: ACCEPTED)
2026-06-12 10:39:54,051 INFO yarn.Client: Application report for application_1781260764017_0001 (state: ACCEPTED)
2026-06-12 10:39:55,055 INFO yarn.Client: Application report for application_1781260764017_0001 (state: ACCEPTED)
2026-06-12 10:39:55,699 INFO cluster.YarnClientSchedulerBackend: Add WebUI Filter. org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter, Map(PROXY_HOSTS -> algo-1, PROXY_URI_BASES -> http://algo-1:8088/proxy/application_1781260764017_0001), /proxy/application_1781260764017_0001
2026-06-12 10:39:56,060 INFO yarn.Client: Application report for application_1781260764017_0001 (state: RUNNING)
2026-06-12 10:39:56,068 INFO yarn.Client: 
#011 client token: N/A
#011 diagnostics: N/A
#011 ApplicationMaster host: 10.0.203.147
#011 ApplicationMaster RPC port: -1
#011 queue: default
#011 start time: 1781260788890
#011 final status: UNDEFINED
#011 tracking 

2026-06-12 10:40:01,832 INFO cluster.YarnSchedulerBackend$YarnDriverEndpoint: Registered executor NettyRpcEndpointRef(spark-client://Executor) (10.0.203.147:48616) with ID 1,  ResourceProfileId 0
2026-06-12 10:40:02,110 INFO storage.BlockManagerMasterEndpoint: Registering block manager algo-1:36143 with 5.9 GiB RAM, BlockManagerId(1, algo-1, 36143, None)


2026-06-12 10:40:13,134 INFO cluster.YarnClientSchedulerBackend: SchedulerBackend is ready for scheduling beginning after waiting maxRegisteredResourcesWaitingTime: 30000000000(ns)
2026-06-12 10:40:13,399 WARN spark.SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/tmp' appears to be on the local filesystem.
2026-06-12 10:40:13,517 INFO internal.SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
2026-06-12 10:40:13,530 INFO internal.SharedState: Warehouse path is 'file:/usr/spark-3.3.0/spark-warehouse'.
2026-06-12 10:40:14,777 INFO datasources.InMemoryFileIndex: It took 61 ms to list leaf files for 1 paths.
2026-06-12 10:40:14,986 INFO memory.MemoryStore: Block broadcast_0 stored as values in memory (estimated size 416.9 KiB, free 1458.2 MiB)
2026-06-12 10:40:15,358 INFO memory.MemoryStore: Block broadcast_0_piece0 stored as bytes in memory (estimated siz

2026-06-12 10:40:23,580 INFO storage.BlockManagerInfo: Removed broadcast_3_piece0 on 10.0.203.147:34811 in memory (size: 8.0 KiB, free: 1458.6 MiB)
2026-06-12 10:40:23,603 INFO storage.BlockManagerInfo: Removed broadcast_3_piece0 on algo-1:36143 in memory (size: 8.0 KiB, free: 5.9 GiB)
2026-06-12 10:40:23,723 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-06-12 10:40:23,922 INFO scheduler.DAGScheduler: Registering RDD 16 (collect at AnalysisRunner.scala:326) as input to shuffle 0
2026-06-12 10:40:23,929 INFO scheduler.DAGScheduler: Got map stage job 2 (collect at AnalysisRunner.scala:326) with 1 output partitions
2026-06-12 10:40:23,930 INFO scheduler.DAGScheduler: Final stage: ShuffleMapStage 2 (collect at AnalysisRunner.scala:326)
2026-06-12 10:40:23,931 INFO scheduler.DAGScheduler: Parents of final stage: List()
2026-06-12 10:40:23,934 INFO scheduler.DAGSchedu

2026-06-12 10:40:33,094 INFO scheduler.TaskSetManager: Finished task 0.0 in stage 23.0 (TID 18) in 93 ms on algo-1 (executor 1) (1/1)
2026-06-12 10:40:33,095 INFO cluster.YarnScheduler: Removed TaskSet 23.0, whose tasks have all completed, from pool 
2026-06-12 10:40:33,096 INFO scheduler.DAGScheduler: ResultStage 23 (treeReduce at KLLRunner.scala:107) finished in 0.109 s
2026-06-12 10:40:33,097 INFO scheduler.DAGScheduler: Job 16 is finished. Cancelling potential speculative or zombie tasks for this job
2026-06-12 10:40:33,097 INFO cluster.YarnScheduler: Killing all running tasks in stage 23: Stage finished
2026-06-12 10:40:33,097 INFO scheduler.DAGScheduler: Job 16 finished: treeReduce at KLLRunner.scala:107, took 0.114808 s
2026-06-12 10:40:33,283 INFO codegen.CodeGenerator: Code generated in 38.622861 ms
2026-06-12 10:40:33,290 INFO scheduler.DAGScheduler: Registering RDD 104 (collect at AnalysisRunner.scala:326) as input to shuffle 7
2026-06-12 10:40:33,291 INFO scheduler.DAGSched

<i class="fas fa-sticky-note" style="color:#ec7211"></i> **NOTE:** This code returns a lengthy response. You can ignore any warnings or error messages.

<i class="fas fa-sticky-note" style="color:#ec7211"></i> **NOTE:** When the cell completes, a message is returned that looks like *2025-10-11 12:13:14,156 - DefaultDataAnalyzer - INFO - Spark job completed*.

Now, search for the *constraints.json* and *statistics.json* files to see where they are located.

In [10]:
#explore-generated-constraints-and-statistics
s3_client = boto3.Session().client("s3")
result = s3_client.list_objects(Bucket=bucket, Prefix=baseline_results_prefix)
report_files = [report_file.get("Key") for report_file in result.get("Contents")]
print("Found Files:")
print("\n ".join(report_files))

Found Files:
sagemaker/abalone/baselining/results/constraints.json
 sagemaker/abalone/baselining/results/statistics.json


Next, view the generated statistics.

In [11]:
#view-statistics
baseline_job = my_default_monitor.latest_baselining_job
schema_df = pd.json_normalize(baseline_job.baseline_statistics().body_dict["features"])
schema_df.head(10)

,name,inferred_type,numerical_statistics.common.num_present,numerical_statistics.common.num_missing,numerical_statistics.mean,numerical_statistics.sum,numerical_statistics.std_dev,numerical_statistics.min,numerical_statistics.max,numerical_statistics.approximate_num_distinct_values,numerical_statistics.completeness,numerical_statistics.distribution.kll.buckets,numerical_statistics.distribution.kll.sketch.parameters.c,numerical_statistics.distribution.kll.sketch.parameters.k,numerical_statistics.distribution.kll.sketch.data
0,rings,Integral,120,0,9.991667e+00,1.199000e+03,2.378711,4.000000,17.000000,13,1.0,"[{'lower_bound': 4.0, 'upper_bound': 5.3, 'cou...",0.64,2048.0,"[[11.0, 12.0, 10.0, 10.0, 13.0, 11.0, 11.0, 10..."
1,length,Fractional,120,0,-2.500000e-11,-3.000000e-09,1.000000,-3.132857,1.792725,60,1.0,"[{'lower_bound': -3.13285749, 'upper_bound': -...",0.64,2048.0,"[[0.842173735, 1.40386293, 1.187828624, 0.9285..."
2,diameter,Fractional,120,0,1.666667e-11,2.000000e-09,1.000000,-3.116738,1.574943,54,1.0,"[{'lower_bound': -3.116738385, 'upper_bound': ...",0.64,2048.0,"[[0.860991542, 1.523946529, 0.962984617, 0.962..."
3,height,Fractional,120,0,-8.326673e-17,-9.992007e-15,1.000000,-2.590157,1.918424,29,1.0,"[{'lower_bound': -2.590157076, 'upper_bound': ...",0.64,2048.0,"[[0.688811003, 1.645176692, 1.098682012, 1.235..."
4,whole_weight,Fractional,120,0,1.666665e-11,1.999998e-09,1.000000,-1.777180,3.316178,119,1.0,"[{'lower_bound': -1.777179983, 'upper_bound': ...",0.64,2048.0,"[[0.797789649, 2.556957875, 1.439731739, 1.895..."
5,shucked_weight,Fractional,120,0,1.666675e-11,2.000010e-09,1.000000,-1.864274,3.056230,119,1.0,"[{'lower_bound': -1.864273528, 'upper_bound': ...",0.64,2048.0,"[[0.819637499, 2.065165072, 1.808023895, 1.861..."
6,viscera_weight,Fractional,120,0,-1.666667e-11,-2.000000e-09,1.000000,-1.751453,2.736338,101,1.0,"[{'lower_bound': -1.751452951, 'upper_bound': ...",0.64,2048.0,"[[0.854509788, 2.736337747, 1.315742131, 1.800..."
7,shell_weight,Fractional,120,0,1.666669e-11,2.000003e-09,1.000000,-1.849493,2.974287,75,1.0,"[{'lower_bound': -1.849493312, 'upper_bound': ...",0.64,2048.0,"[[0.46480584, 2.137793179, 0.848198772, 0.8481..."
8,sex_m,Integral,120,0,4.083333e-01,4.900000e+01,0.491525,0.000000,1.000000,2,1.0,"[{'lower_bound': 0.0, 'upper_bound': 0.1, 'cou...",0.64,2048.0,"[[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0,..."
9,sex_f,Integral,120,0,1.833333e-01,2.200000e+01,0.386940,0.000000,1.000000,2,1.0,"[{'lower_bound': 0.0, 'upper_bound': 0.1, 'cou...",0.64,2048.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


The statistics table shows each feature with its corresponding summary statistics, including the mean, standard deviation, min, max, and other important details.

Finally, view the generated constraints.

In [12]:
#view-constraints
constraints_df = pd.json_normalize(
    baseline_job.suggested_constraints().body_dict["features"]
)
constraints_df.head(10)

,name,inferred_type,completeness,num_constraints.is_non_negative
0,rings,Integral,1.0,True
1,length,Fractional,1.0,False
2,diameter,Fractional,1.0,False
3,height,Fractional,1.0,False
4,whole_weight,Fractional,1.0,False
5,shucked_weight,Fractional,1.0,False
6,viscera_weight,Fractional,1.0,False
7,shell_weight,Fractional,1.0,False
8,sex_m,Integral,1.0,True
9,sex_f,Integral,1.0,True


The constraints table shows the inferred type of each feature, the record completeness (in this case 1.0 for all features because the file has no missing values), and the fields that have no non-negative values. 

Now that you have a baseline created and have viewed the statistics and constraints, create a Model Monitor data quality monitoring job to track new inference records against the baseline.

# Task 2.5: Create a Model Monitor data quality monitoring job

After you create your baseline, you can call the *create_monitoring_schedule()* method of the *DefaultModelMonitor* class instance to schedule an hourly data quality monitor.

In this task, you analyze and monitor the data with a data quality monitoring job.

First, use the *create_monitoring_schedule()* method to schedule an hourly data quality monitoring schedule. 

In [13]:
#create-monitoring-schedule
s3_bucket = boto3.Session().resource("s3").Bucket(bucket)
s3_bucket.Object(code_prefix + "/postprocessor.py").upload_file(f"{repo_path}/python/postprocessor.py")

mon_schedule_name = f"model-monitor-schedule-{datetime.now():%Y-%m-%d-%H-%M-%S}"

my_default_monitor.create_monitoring_schedule(
    monitor_schedule_name=mon_schedule_name,
    endpoint_input=predictor.endpoint_name,
    post_analytics_processor_script=s3_code_postprocessor_uri,
    output_s3_uri=s3_report_path,
    statistics=my_default_monitor.baseline_statistics(),
    constraints=my_default_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: model-monitor-schedule-2026-06-12-10-54-56


Next, send some artificial traffic to the endpoint for the monitoring job to be able to generate the violations report. To simulate data drift, use a set of skewed data. The skewed data, when compared against the baseline, throws an alert with the automated alert triggering system.

In [14]:
#send-artificial-traffic
endpoint_name = predictor.endpoint_name
runtime_client = sm_session.sagemaker_runtime_client
limit = 200
i = 0

# repeating code from above to run this section independently
def invoke_endpoint(ep_name, file_name, runtime_client):
    i = 0
    with open(file_name, "r") as f:
        for row in f:
            (label, payload) = row.strip("\n").split(",", 1)  

            response = runtime_client.invoke_endpoint(
                EndpointName=ep_name, ContentType="text/csv", Body=payload
            )
            response["Body"].read()
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            time.sleep(0.5)


invoke_endpoint(endpoint_name, f"{repo_path}/data/abalone_data_skewed.csv", runtime_client)
print("\nDone!")

.

.

.

.

.

...

.

.

.

.


Done!


Use *describe_schedule* to view the schedule you just created.

In [15]:
#model-monitor-schedule-status
desc_schedule_result = my_default_monitor.describe_schedule()
print("Schedule status: {}".format(desc_schedule_result["MonitoringScheduleStatus"]))

Schedule status: Scheduled


The monitor schedule starts jobs at the previously specified hourly interval. Even for an hourly schedule, SageMaker AI has a buffer period of 20 minutes to schedule your execution. You might see your execution start anywhere from 0 to 20 minutes from the hour boundary. This is expected and done for load balancing in the backend.

This execution takes approximately one hour to be able to generate the violations report. For the purpose of the lab, the next cells have code snippets for you to view and sample output is shared for reference. In the last step of this task, you view the violations report from a file that was generated and pre-loaded from an earlier monitoring run.

When the execution finishes, SageMaker AI reports the status of the latest completed or failed execution. 

Here are the possible terminal states:
- **Completed** - The monitoring execution completed and no issues were found in the violations report. 
- **CompletedWithViolations** - The execution completed, but constraint violations were detected. 
- **Failed** - The monitoring execution failed, maybe due to client error (perhaps incorrect role permissions) or infrastructure issues. Further examination of FailureReason and ExitMessage is necessary to identify what exactly happened. 
- **Stopped** - The job exceeded max runtime or was manually stopped.


If you want to list and view the current status of an execution, you can use code similar to this:

```python 
# list the current execution
mon_executions = my_default_monitor.list_executions()
print(
    "We created a hourly schedule above that begins executions ON the hour (plus 0-20 min buffer.\nWe will have to wait for an hour..."
)

while len(mon_executions) == 0:
    print("Waiting for the first execution to happen...")
    time.sleep(60)
    mon_executions = my_default_monitor.list_executions()
```



```python
# Latest execution status
latest_execution = mon_executions[-1]  # Latest execution's index is -1, second to last is -2, etc
time.sleep(60)
latest_execution.wait(logs=False)

print("Latest execution status: {}".format(latest_execution.describe()["ProcessingJobStatus"]))
print("Latest execution result: {}".format(latest_execution.describe()["ExitMessage"]))

latest_job = latest_execution.describe()
if latest_job["ProcessingJobStatus"] != "Completed":
    print(
        "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
    )
```

The following is the expected output when the latest execution of the monitoring job completes.

```bash
!Latest execution status: Completed

Latest execution result: CompletedWithViolations: Job completed successfully with 8 violations.
```

To list the generated violation report, you can use code similar to this:

```python

from urllib.parse import urlparse

report_uri = latest_execution.output.destination
s3uri = urlparse(report_uri)
report_bucket = s3uri.netloc
report_key = s3uri.path.lstrip("/")
s3_client = boto3.Session().client("s3")
result = s3_client.list_objects(Bucket=report_bucket, Prefix=report_key)
report_files = [report_file.get("Key") for report_file in result.get("Contents")]
print("Found Report Files:")
print("\n ".join(report_files))
```

The following is the listing the report files.

```bash
Found Report Files:
sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/constraint_violations.json

sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/constraints.json

sagemaker/abalone/reports/Abalone-2023-09-20-17-05-28/model-monitor-schedule-2023-09-20-17-41-22/2023/09/20/18/statistics.json
```

To list violations compare to the baseline, you can use code similar to this:

```python
violations = my_default_monitor.latest_monitoring_constraint_violations()
pd.set_option("display.max_colwidth", None)
constraints_df = pd.json_normalize(violations.body_dict["violations"])
constraints_df.head(10)
```

Since the execution of the monitoring job you started above will not finish for 60-80 minutes, view a violations report from a file that was generated and pre-loaded from an earlier monitoring run.

In [16]:
#print-violations-report
pd.set_option('display.max_colwidth', None)
violations = json.load(open(f'{repo_path}/data/violations.json'))
constraints_df=pd.json_normalize(violations, record_path=['violations'])
constraints_df.head(10)

,feature_name,constraint_check_type,description
0,rings,data_type_check,"Data type match requirement is not met. Expected data type: Integral, Expected match: 100.0%. Observed: Only 0.0% of data is Integral."
1,sex_f,data_type_check,"Data type match requirement is not met. Expected data type: Integral, Expected match: 100.0%. Observed: Only 62.616822429906534% of data is Integral."
2,sex_i,data_type_check,"Data type match requirement is not met. Expected data type: Integral, Expected match: 100.0%. Observed: Only 62.616822429906534% of data is Integral."
3,sex_m,data_type_check,"Data type match requirement is not met. Expected data type: Integral, Expected match: 100.0%. Observed: Only 62.616822429906534% of data is Integral."
4,length,baseline_drift_check,Baseline drift distance: 0.17354640931273702 exceeds threshold: 0.1
5,diameter,baseline_drift_check,Baseline drift distance: 0.14440240796849557 exceeds threshold: 0.1
6,whole_weight,baseline_drift_check,Baseline drift distance: 0.11630482464065406 exceeds threshold: 0.1
7,shell_weight,baseline_drift_check,Baseline drift distance: 0.11238588915222741 exceeds threshold: 0.1


The violations report shows eight violations from the skewed data file, with four **data_type_check** violations for **rings**, **sex_f**, **sex_i**, and **sex_m**, and four **baseline_drift_check** violations for **length**, **diameter**, **whole_weight**, and **shell_weight**.

Take a moment to view the description for each violation. Notice that the data type matches were not correct for some of the features. Also, notice that the baseline drift distance exceeded the *0.1* threshold for the four reported features.

<i class="fas fa-sticky-note" style="color:#ff6633" aria-hidden="true"></i> **Note:** When you created a baseline job, it produced constraints.json and statistics.json files. In the *constraints.json* file, the *comparison_threshold* is set to *0.1* by default. To learn more about constraints.json file, refer [Schema for Constraints](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-byoc-constraints.html).

## Task 2.6: Create a CloudWatch alarm

When data drift happens, it is helpful to get notifications so you can address any issues. A notification or alarm can also trigger automatic model retraining to address changes that might be occurring with your inference data.

In this task, you learn how to create alarm and enable notifications to know when data drifts away from baseline data.

First, find your current Amazon Simple Notification Service(Amazon SNS) topics by using **list_topics**.

In [17]:
#list-topics
client = boto3.client('sns')
topics_list= client.list_topics()
print(topics_list)

{'Topics': [{'TopicArn': 'arn:aws:sns:us-west-2:105806319240:MetricAlertSNSTopic'}], 'ResponseMetadata': {'RequestId': '154eb8cf-5305-5e8b-a289-da6abbabb320', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '154eb8cf-5305-5e8b-a289-da6abbabb320', 'date': 'Fri, 12 Jun 2026 10:58:54 GMT', 'content-type': 'text/xml', 'content-length': '384', 'connection': 'keep-alive'}, 'RetryAttempts': 0}}


Next, set the **sns_notifications_topic** variable with the topic ARN value you found in the prior cell.

In [18]:
#set-variables
topic_details = pd.json_normalize(topics_list['Topics'])
topic_arn = topic_details['TopicArn']
print (topic_arn[0])
sns_notifications_topic = topic_arn[0]

arn:aws:sns:us-west-2:105806319240:MetricAlertSNSTopic


Finally, create an alarm using **put_metric_alarm** that triggers a notification when the feature diameter drifts away from the baseline and retrain the model automatically. 

You use the built-in SageMaker AI Model Monitor container for CloudWatch metrics. SageMaker AI emits the metrics for each feature observed in the dataset in the */aws/sagemaker/Endpoints/data-metric* namespace with *EndpointName* and *ScheduleName* dimensions

In [19]:
#trigger-cloudwatch-alarm-when-it-drifts-from-baseline
cw_client = boto3.Session().client('cloudwatch')

alarm_name = 'BASELINE_DRIFT_FEATURE_DIAMETER'
alarm_desc = 'Trigger a CloudWatch alarm when the feature diameter drifts away from the baseline'
feature_diameter_drift_threshold = 0.1  # Setting this threshold purposefully low to see the alarm quickly.
metric_name = 'feature_baseline_drift_diameter'
namespace = 'aws/sagemaker/Endpoints/data-metrics'

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    AlarmActions=[sns_notifications_topic],
    MetricName=metric_name,
    Namespace=namespace,
    Statistic='Sum',
    Dimensions=[
        {
            'Name': 'Endpoint',
            'Value': endpoint_name
        },
        {
            'Name': 'MonitoringSchedule',
            'Value': mon_schedule_name
        }
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=feature_diameter_drift_threshold,
    ComparisonOperator='GreaterThanOrEqualToThreshold',
    TreatMissingData='breaching'
)


{'ResponseMetadata': {'RequestId': '71c3e400-f5cf-484d-a5ce-2275d4a08d68',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '71c3e400-f5cf-484d-a5ce-2275d4a08d68',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '0',
   'date': 'Fri, 12 Jun 2026 10:59:02 GMT'},
  'RetryAttempts': 0}}

You created a CloudWatch alarm. You can use this alarm to notify you of any data drift issues and trigger automatic model retraining. However, this alarm is set to become an alert state immediately after creation.

## Continue to Task 3

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file.
- Return to the lab session and continue with **Task 3**.